# Retriever 실험

저장된 `vectorstore`와 `split_documents`를 불러와  
`Naive / Dense / Hybrid / Reranker` 리트리버를 같은 평가셋에서 비교합니다.

평가 지표는 다음 4가지를 사용합니다.

| 지표 | 설명 |
|------|------|
| **Recall@K** | 정답 문서가 상위 K개 검색 결과 안에 포함되는 비율 |
| **Precision@K** | 상위 K개 검색 결과 중 실제로 정답에 해당되는 문서 비율 |
| **MRR (Mean Reciprocal Rank)** | 정답 문서가 몇 번째에 등장했는지의 역수 평균 |
| **nDCG (Normalized Discounted Cumulative Gain)** | 순위의 품질을 점수 기반으로 평가하는 지표 |


In [11]:
# [1] 저장된 vectorstore / split_documents 로드
import sys
import pickle
from pathlib import Path

PROJECT_ROOT = Path('/Users/apple/Team2-RAG-Project')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from src.embedding import create_embeddings

load_dotenv(dotenv_path=PROJECT_ROOT / '.env')
VECTORSTORE_DIR = PROJECT_ROOT / 'data/vectorstore/faiss_openai'
SPLIT_DOCS_PATH = PROJECT_ROOT / 'data/vectorstore/split_documents.pkl'

if not VECTORSTORE_DIR.exists():
    raise FileNotFoundError(f'vectorstore 경로 없음: {VECTORSTORE_DIR}')
if not SPLIT_DOCS_PATH.exists():
    raise FileNotFoundError(f'split_documents 파일 없음: {SPLIT_DOCS_PATH}')

embeddings = create_embeddings(model='text-embedding-3-small', chunk_size=100)
vectorstore = FAISS.load_local(
    str(VECTORSTORE_DIR),
    embeddings,
    allow_dangerous_deserialization=True,
)

with open(SPLIT_DOCS_PATH, 'rb') as f:
    split_documents = pickle.load(f)

print('준비 완료')
print('vectorstore:', VECTORSTORE_DIR)
print('split_documents 개수:', len(split_documents))


준비 완료
vectorstore: /Users/apple/Team2-RAG-Project/data/vectorstore/faiss_openai
split_documents 개수: 10915


In [12]:
# [2] 리트리버 4종 생성
from retrievers.retriever_naive import build_retriever_naive
from retrievers.retriever_dense import build_retriever_dense
from retrievers.retriever_hybrid import build_retriever_hybrid
from retrievers.retriever_reranker import build_retriever_reranker

RETRIEVER_K = 10

retriever_map = {
    'naive': build_retriever_naive(vectorstore, k=RETRIEVER_K),
    'dense': build_retriever_dense(vectorstore, k=RETRIEVER_K),
    'hybrid': build_retriever_hybrid(
        vectorstore,
        split_documents,
        dense_k=RETRIEVER_K,
    ),
    'reranker': build_retriever_reranker(
        vectorstore,
        first_k=20,
        top_n=RETRIEVER_K,
    ),
}
list(retriever_map.keys())


['naive', 'dense', 'hybrid', 'reranker']

In [13]:
# [3] 평가 함수(Recall@k, Hit@k, MRR)
import json
import re
import unicodedata
import pandas as pd

EVAL_PATH = PROJECT_ROOT / 'evaluation/test_questions.json'
dataset = json.loads(EVAL_PATH.read_text(encoding='utf-8'))
print('평가 문항 수:', len(dataset))

KS = [1, 3, 5, 10]

def normalize_file_name(name):
    s = unicodedata.normalize('NFC', str(name or '')).lower().strip()
    s = re.sub(r'\.(pdf|hwp|hwpx|jsonl)$', '', s)
    s = re.sub(r'[\s_\-\(\)\[\]\{\}·ㆍ\.,!?:;`~|]', '', s)
    return s

def to_page_1based(page):
    try:
        p = int(page)
    except Exception:
        return None
    return p + 1 if p >= 0 else None

def gold_pairs(item):
    pairs = set()
    for ev in item.get('evidence', []):
        sf = normalize_file_name(ev.get('source_file'))
        pg = ev.get('page')
        if sf:
            pairs.add((sf, int(pg) if pg is not None else None))
    return pairs

def pred_pairs(docs):
    pairs = []
    for d in docs:
        md = getattr(d, 'metadata', {}) or {}
        sf = normalize_file_name(md.get('source_file') or md.get('source') or '')
        pg = to_page_1based(md.get('page'))
        pairs.append((sf, pg))
    return pairs

def evaluate_retriever(retriever, data, ks=KS):
    rows = []
    for item in data:
        q = item['question']
        gold = gold_pairs(item)
        docs = retriever.invoke(q)
        pred = pred_pairs(docs)
        row = {'id': item.get('id'), 'question': q, 'gold_evidence_count': len(gold)}
        rr = 0.0
        for rank, pair in enumerate(pred, start=1):
            if pair in gold:
                rr = 1.0 / rank
                break
        row['RR'] = rr
        for k in ks:
            topk = pred[:k]
            row[f'Hit@{k}'] = 1 if any(p in gold for p in topk) else 0
            if len(gold) == 0:
                row[f'Recall@{k}'] = 0.0
            else:
                row[f'Recall@{k}'] = len(set(topk) & gold) / len(gold)
        rows.append(row)
    detail_df = pd.DataFrame(rows)
    summary = {'MRR': detail_df['RR'].mean()}
    for k in ks:
        summary[f'Hit@{k}'] = detail_df[f'Hit@{k}'].mean()
        summary[f'Recall@{k}'] = detail_df[f'Recall@{k}'].mean()
    return pd.DataFrame([summary]), detail_df


평가 문항 수: 30


In [14]:
# [4] 리트리버별 평가 실행
all_summary = []
detail_results = {}

for name, r in retriever_map.items():
    print(f'평가 중: {name}')
    s_df, d_df = evaluate_retriever(r, dataset, ks=KS)
    s_df.insert(0, 'retriever', name)
    all_summary.append(s_df)
    detail_results[name] = d_df

summary_df = pd.concat(all_summary, ignore_index=True)
summary_df = summary_df.sort_values('MRR', ascending=False).reset_index(drop=True)
summary_df


평가 중: naive
평가 중: dense
평가 중: hybrid
평가 중: reranker


,retriever,MRR,Hit@1,Recall@1,Hit@3,Recall@3,Hit@5,Recall@5,Hit@10,Recall@10
0,hybrid,0.152302,0.100000,0.100000,0.133333,0.133333,0.233333,0.233333,0.366667,0.366667
1,reranker,0.146111,0.066667,0.066667,0.166667,0.166667,0.233333,0.233333,0.333333,0.333333
2,naive,0.140079,0.066667,0.066667,0.166667,0.166667,0.233333,0.233333,0.366667,0.366667
3,dense,0.140079,0.066667,0.066667,0.166667,0.166667,0.233333,0.233333,0.366667,0.366667


In [15]:
# [5] 문항별 상세 확인
target = 'hybrid'
cols = ['id', 'question', 'RR'] + [f'Hit@{k}' for k in KS] + [f'Recall@{k}' for k in KS]
detail_results[target][cols].head(30)


,id,question,RR,Hit@1,Hit@3,Hit@5,Hit@10,Recall@1,Recall@3,Recall@5,Recall@10
0,1,봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총 얼마로 잡혀 있나요? 주관 ...,0.250000,0,0,1,1,0.0,0.0,1.0,1.0
1,2,이번 봉화군 재난위험지역 경보발령 사업에서 활용하는 무선 통신 방식은 구체적으로 무...,0.000000,0,0,0,0,0.0,0.0,0.0,0.0
2,3,봉화군 재난 사업용 하드웨어 규격 중에서 ECR-02 무선경보장치 스피커의 출력 기...,0.000000,0,0,0,0,0.0,0.0,0.0,0.0
3,4,고려대 차세대 학사시스템 구축 사업의 전체 예산 규모와 연도별 분할 지급 비율이 어...,1.000000,1,1,1,1,1.0,1.0,1.0,1.0
4,5,고려대 포털 고도화 작업 중에 기존 'AI선배' 서비스의 아키텍처는 어떤 방향으로 ...,0.142857,0,0,0,1,0.0,0.0,0.0,1.0
5,6,고려대 학사행정시스템을 구축할 때 서울과 세종 캠퍼스별 처리에 대한 별도의 지침이 ...,0.000000,0,0,0,0,0.0,0.0,0.0,0.0
6,7,중앙의료원 응급의료 상황관리시스템 구축 사업에서 광역상황실이 신규로 설치되는 4개 ...,1.000000,1,1,1,1,1.0,1.0,1.0,1.0
7,8,국립중앙의료원 인프라 요건 중 DB 서버(Server-001)가 갖추어야 할 최소 ...,0.000000,0,0,0,0,0.0,0.0,0.0,0.0
8,9,응급환자 전원 지원 시스템의 핵심 업무 중 '상황의사'는 주로 어떤 역할을 수행하게...,0.000000,0,0,0,0,0.0,0.0,0.0,0.0
9,10,나노종합기술원의 스마트 팹 설비온라인 시스템 구축 용역은 계약 후 기간이 총 얼마나...,1.000000,1,1,1,1,1.0,1.0,1.0,1.0


### 결론

- **Hybrid retriever가 전체적으로 가장 우수**했습니다.  
  - MRR이 가장 높고(`0.1523`), Hit@1/Recall@1도 최고(`0.10`)였습니다.
- **Reranker는 2위 성능**을 보였습니다.  
  - MRR은 `0.1461`로 높았지만, Hit@10/Recall@10은 `0.3333`으로 Hybrid(`0.3667`)보다 낮았습니다.
- **Naive와 Dense는 동일 성능**이었습니다.  
  - 모든 지표가 동일해 현재 설정에서는 동작 차이가 거의 없었습니다.
- 따라서 현재 실험 기준에서는 **Hybrid를 기본 retriever로 채택**하는 것이 가장 합리적입니다.
